# 📊 Notebook 02 — EDA (Exploratory Data Analysis)

**Input:** `data/synthetic/helmet_imu_raw.csv`  
**Goal:** Understand the raw data — distributions, correlations, noise, separability between classes.  
**Output:** Insights + cleaned understanding before we do feature extraction.

---

EDA answers:
- Are the classes visually separable?
- Which sensor axes carry the most information?
- Are there missing values or outliers?
- What window size makes sense?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

df = pd.read_csv('../data/synthetic/helmet_imu_raw.csv')
print(f'Shape: {df.shape}')
display(df.head())

In [ ]:
# ── Missing values + basic info ──────────────────────────────────
print('Missing values:')
print(df.isnull().sum())
print('\nLabel counts:')
print(df['label_name'].value_counts())
print('\nDescriptive stats:')
display(df[['ax','ay','az','gx','gy','gz']].describe().round(3))

In [ ]:
# ── Distribution per axis per class ─────────────────────────────
COLORS = {0: '#2ecc71', 1: '#f1c40f', 2: '#e67e22', 3: '#e74c3c'}
NAMES  = {0: 'Normal', 1: 'Pothole', 2: 'SuddenBrake', 3: 'Crash'}
features = ['ax','ay','az','gx','gy','gz']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.patch.set_facecolor('#0d0d0d')

for i, feat in enumerate(features):
    ax = axes[i//3, i%3]
    ax.set_facecolor('#1a1a2e')
    for lbl in range(4):
        data = df[df['label'] == lbl][feat]
        ax.hist(data, bins=60, alpha=0.55, color=COLORS[lbl],
                label=NAMES[lbl], density=True, edgecolor='none')
    ax.set_title(feat, color='#e0e0e0', fontsize=11)
    ax.tick_params(colors='#aaaaaa')
    ax.grid(True, color='#222244', linewidth=0.4)
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')
    if i == 0:
        patches = [mpatches.Patch(color=COLORS[l], label=NAMES[l]) for l in range(4)]
        ax.legend(handles=patches, facecolor='#1a1a2e', edgecolor='#444466', labelcolor='white', fontsize=8)

fig.suptitle('Feature Distributions by Class', color='white', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Violin plots — better than box plots for IMU ────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0d0d0d')

for i, feat in enumerate(['az', 'ax', 'gy']):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')
    data = [df[df['label'] == l][feat].sample(min(2000, len(df[df['label']==l]))).values for l in range(4)]
    parts = ax.violinplot(data, positions=range(4), showmedians=True, showextrema=True)
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(COLORS[j])
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('white')
    parts['cmaxes'].set_color('#888888')
    parts['cmins'].set_color('#888888')
    parts['cbars'].set_color('#888888')
    ax.set_xticks(range(4))
    ax.set_xticklabels([NAMES[l] for l in range(4)], color='#aaaaaa', rotation=15, fontsize=8)
    ax.set_title(feat, color='#e0e0e0', fontsize=11)
    ax.tick_params(colors='#aaaaaa')
    ax.grid(True, axis='y', color='#222244', linewidth=0.4)
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')

fig.suptitle('Violin Plots — Key Axes by Class', color='white', fontsize=13)
plt.tight_layout()
plt.show()

print('Observations:')
print('  az: Normal/Pothole/Brake ~9.81. Crash loses gravity — distribution shifts.')
print('  ax: SuddenBrake strongly negative. Others near zero.')
print('  gy: Brake has pitch-forward (positive). Crash has massive spread.')

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────
corr = df[['ax','ay','az','gx','gy','gz','label']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#1a1a2e')

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=ax, linewidths=0.5, linecolor='#1a1a2e',
            annot_kws={'size': 9},
            vmin=-1, vmax=1)

ax.set_title('Correlation Matrix (including label)', color='white', fontsize=12)
ax.tick_params(colors='#e0e0e0')
plt.tight_layout()
plt.show()
print('\nlabel correlations:')
print(corr['label'].drop('label').sort_values())

In [ ]:
# ── Acceleration + Gyro magnitude over time (one session) ────────
from src.data_generator import LABEL_MAP

COLOR_MAP = {0: '#2ecc71', 1: '#f1c40f', 2: '#e67e22', 3: '#e74c3c'}

# Pick a session with a crash
crash_sessions = df[df['label'] == 3]['session_id'].unique()
sid = crash_sessions[2]  # 3rd crash session for variety
sess = df[df['session_id'] == sid].reset_index(drop=True)

t = sess['timestamp'].values
accel_mag = np.sqrt(sess['ax']**2 + sess['ay']**2 + sess['az']**2)
gyro_mag  = np.sqrt(sess['gx']**2 + sess['gy']**2 + sess['gz']**2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.patch.set_facecolor('#0d0d0d')

for ax_plot, data, title in [(ax1, accel_mag, 'Accel Magnitude (m/s²)'),
                               (ax2, gyro_mag,  'Gyro Magnitude (°/s)')]:
    ax_plot.set_facecolor('#1a1a2e')
    for i in range(len(t)-1):
        ax_plot.plot([t[i], t[i+1]], [data.iloc[i], data.iloc[i+1]],
                    color=COLOR_MAP[sess['label'].iloc[i]], linewidth=2, alpha=0.9)
    ax_plot.set_ylabel(title, color='#aaaaaa')
    ax_plot.tick_params(colors='#aaaaaa')
    ax_plot.grid(True, color='#222244', linewidth=0.5)
    for spine in ax_plot.spines.values(): spine.set_edgecolor('#333355')

patches = [mpatches.Patch(color=COLOR_MAP[l], label=LABEL_MAP[l]) for l in range(4)]
fig.legend(handles=patches, loc='upper right', ncol=4,
           facecolor='#1a1a2e', edgecolor='#444466', labelcolor='white', fontsize=9)
fig.suptitle(f'Session {sid} — Magnitude over Time (colored by event)', color='white', fontsize=13)
ax2.set_xlabel('Timestamp', color='#aaaaaa')
plt.tight_layout()
plt.show()

In [ ]:
# ── Window size analysis ─────────────────────────────────────────
print('=== Window Size Decision ===')
print()
print('Observations from EDA:')
print('  - A crash event typically lasts 4–8 timesteps (impact spike)')
print('  - Post-crash chaos lasts 10–20 timesteps')
print('  - A sudden brake ramps over ~15 timesteps')
print()
print('Decision: window_size = 20, stride = 10')
print('  → Captures full brake + crash patterns')
print('  → 50% overlap gives dense coverage')
print('  → At 50Hz real MPU6050: window = 0.4s (reasonable pre-impact window)')
print()
print('Next: notebooks/03_feature_engineering.ipynb')